# Geo-Holdout Experiment Design and Power Analysis

## Measurement 360 Portfolio

This notebook designs a synthetic geo-holdout experiment using the eligible
United States regions identified in the BigQuery analytical warehouse.

### Objectives

1. Load and validate eligible PRE-period geo-week observations.
2. Evaluate regional stability and scale.
3. Match comparable treatment and control regions using PRE-period outcomes.
4. Estimate pre-treatment balance and statistical power.
5. Freeze the experiment design before examining TEST-period outcomes.

### Data boundary

- Observed source: Public GA4 ecommerce data.
- Eligible regions: 20 United States regions.
- PRE period: 8 weeks.
- TEST period: 5 weeks.
- The TEST-period outcomes are not used during matching or treatment assignment.
- Treatment effects will be simulated transparently because the public dataset
  does not contain a real randomized media intervention.

In [1]:
from google.colab import auth
from google.cloud import bigquery

import numpy as np
import pandas as pd

PROJECT_ID = "measurement-360-portfolio"
LOCATION = "US"

auth.authenticate_user(project_id=PROJECT_ID)

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION
)

print("Authenticated project:", client.project)
print("Default BigQuery location:", client.location)

Authenticated project: measurement-360-portfolio
Default BigQuery location: US


In [2]:
PRE_PERIOD_QUERY = """
SELECT
    geo_key,
    geo_country,
    geo_region,
    week_start_date,
    week_end_date,
    analysis_week_number,
    period_week_number,
    observed_session_count,
    observed_unique_users,
    observed_engaged_sessions,
    observed_new_user_sessions,
    observed_transaction_count,
    observed_revenue_usd,
    observed_item_quantity,
    observed_transaction_rate,
    observed_revenue_per_session_usd,
    observed_average_order_value_usd,
    pre_session_count,
    pre_transaction_count,
    pre_revenue_usd,
    pre_revenue_coefficient_of_variation,
    data_origin,
    treatment_assignment_status,
    is_synthetic_outcome
FROM
    `measurement-360-portfolio.measurement_360_mart.fct_geo_weekly_outcomes`
WHERE
    meets_initial_eligibility = TRUE
    AND period_name = 'PRE'
ORDER BY
    geo_region,
    week_start_date
"""

job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=50 * 1024 * 1024,
    use_query_cache=True
)

query_job = client.query(
    PRE_PERIOD_QUERY,
    job_config=job_config,
    location=LOCATION
)

pre_df = query_job.to_dataframe()

processed_mb = (query_job.total_bytes_processed or 0) / (1024 ** 2)
billed_mb = (query_job.total_bytes_billed or 0) / (1024 ** 2)

print("Query status:", query_job.state)
print(f"Bytes processed: {processed_mb:.2f} MB")
print(f"Bytes billed: {billed_mb:.2f} MB")
print("Dataframe shape:", pre_df.shape)

Query status: DONE
Bytes processed: 0.15 MB
Bytes billed: 10.00 MB
Dataframe shape: (160, 24)


In [3]:
pre_df["week_start_date"] = pd.to_datetime(pre_df["week_start_date"])
pre_df["week_end_date"] = pd.to_datetime(pre_df["week_end_date"])

display(pre_df.head())

print("Rows:", len(pre_df))
print("Regions:", pre_df["geo_region"].nunique())
print("PRE weeks:", pre_df["week_start_date"].nunique())
print("First PRE week:", pre_df["week_start_date"].min().date())
print("Last PRE week:", pre_df["week_end_date"].max().date())

print("\nRows per region:")
print(
    pre_df.groupby("geo_region")
    .size()
    .value_counts()
    .sort_index()
)

,geo_key,geo_country,geo_region,week_start_date,week_end_date,analysis_week_number,period_week_number,observed_session_count,observed_unique_users,observed_engaged_sessions,...,observed_transaction_rate,observed_revenue_per_session_usd,observed_average_order_value_usd,pre_session_count,pre_transaction_count,pre_revenue_usd,pre_revenue_coefficient_of_variation,data_origin,treatment_assignment_status,is_synthetic_outcome
0,United States | Arizona,United States,Arizona,2020-11-01,2020-11-07,1,1,196,142,160,...,0.015306,0.698980,45.666667,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False
1,United States | Arizona,United States,Arizona,2020-11-08,2020-11-14,2,2,118,90,100,...,0.008475,0.186441,22.000000,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False
2,United States | Arizona,United States,Arizona,2020-11-15,2020-11-21,3,3,172,125,145,...,0.029070,1.709302,58.800000,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False
3,United States | Arizona,United States,Arizona,2020-11-22,2020-11-28,4,4,147,123,129,...,0.006803,0.959184,141.000000,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False
4,United States | Arizona,United States,Arizona,2020-11-29,2020-12-05,5,5,194,152,166,...,0.046392,3.943299,85.000000,1403,42,3520.0,1.150195,PUBLIC_GA4_OBSERVED,UNASSIGNED,False


Rows: 160
Regions: 20
PRE weeks: 8
First PRE week: 2020-11-01
Last PRE week: 2020-12-26

Rows per region:
8    20
Name: count, dtype: int64


In [4]:
required_columns = {
    "geo_key",
    "geo_region",
    "week_start_date",
    "observed_session_count",
    "observed_transaction_count",
    "observed_revenue_usd",
    "pre_revenue_coefficient_of_variation",
    "data_origin",
    "treatment_assignment_status",
    "is_synthetic_outcome"
}

assert required_columns.issubset(pre_df.columns), \
    "One or more required fields are missing."

assert len(pre_df) == 160, \
    "Expected 160 eligible PRE-period region-week rows."

assert pre_df["geo_region"].nunique() == 20, \
    "Expected 20 eligible regions."

assert pre_df["week_start_date"].nunique() == 8, \
    "Expected eight PRE-period weeks."

assert pre_df.groupby("geo_region").size().eq(8).all(), \
    "Every eligible region must contain exactly eight PRE weeks."

assert pre_df["geo_key"].notna().all(), \
    "Every observation must have a geo key."

assert (
    pre_df[
        [
            "observed_session_count",
            "observed_transaction_count",
            "observed_revenue_usd"
        ]
    ] >= 0
).all().all(), "Outcome values cannot be negative."

assert pre_df["data_origin"].eq("PUBLIC_GA4_OBSERVED").all(), \
    "PRE observations must come from observed GA4 data."

assert pre_df["treatment_assignment_status"].eq("UNASSIGNED").all(), \
    "Treatment must remain unassigned during baseline extraction."

assert pre_df["is_synthetic_outcome"].eq(False).all(), \
    "PRE-period observations must not be synthetic."

assert int(pre_df["observed_session_count"].sum()) == 81006, \
    "PRE-period sessions do not reconcile."

assert int(pre_df["observed_transaction_count"].sum()) == 1499, \
    "PRE-period transactions do not reconcile."

assert np.isclose(
    pre_df["observed_revenue_usd"].sum(),
    104458.00,
    atol=0.01
), "PRE-period revenue does not reconcile."

print("PRE-PERIOD EXTRACTION: PASS")

PRE-PERIOD EXTRACTION: PASS


In [5]:
region_summary = (
    pre_df
    .groupby("geo_region", as_index=False)
    .agg(
        pre_weeks=("week_start_date", "nunique"),
        pre_sessions=("observed_session_count", "sum"),
        pre_transactions=("observed_transaction_count", "sum"),
        pre_revenue_usd=("observed_revenue_usd", "sum"),
        average_weekly_revenue_usd=(
            "observed_revenue_usd",
            "mean"
        ),
        weekly_revenue_stddev_usd=(
            "observed_revenue_usd",
            "std"
        ),
        revenue_coefficient_of_variation=(
            "pre_revenue_coefficient_of_variation",
            "first"
        )
    )
    .sort_values(
        ["pre_revenue_usd", "pre_sessions"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(region_summary)

print("Summary rows:", len(region_summary))
print("Summary sessions:", region_summary["pre_sessions"].sum())
print("Summary transactions:", region_summary["pre_transactions"].sum())
print(
    "Summary revenue:",
    round(region_summary["pre_revenue_usd"].sum(), 2)
)

,geo_region,pre_weeks,pre_sessions,pre_transactions,pre_revenue_usd,average_weekly_revenue_usd,weekly_revenue_stddev_usd,revenue_coefficient_of_variation
0,California,8,20964,383,27274.0,3409.250,1630.179285,0.478164
1,Texas,8,7412,165,11680.0,1460.000,775.586230,0.531223
2,Virginia,8,6498,117,7873.0,984.125,797.514610,0.810379
3,Florida,8,4575,86,7084.0,885.500,653.989952,0.738554
4,New York,8,6717,108,5468.0,683.500,405.389055,0.593108
5,New Jersey,8,2948,46,5224.0,653.000,493.880842,0.756326
6,Massachusetts,8,3929,68,4742.0,592.750,326.129575,0.550198
7,North Carolina,8,2471,46,3905.0,488.125,438.166452,0.897652
8,Illinois,8,3757,62,3757.0,469.625,346.690450,0.738228
9,Michigan,8,2464,46,3678.0,459.750,334.639998,0.727874


Summary rows: 20
Summary sessions: 81006
Summary transactions: 1499
Summary revenue: 104458.0


In [6]:
revenue_matrix = (
    pre_df.pivot(
        index="week_start_date",
        columns="geo_region",
        values="observed_revenue_usd"
    )
    .sort_index()
)

session_matrix = (
    pre_df.pivot(
        index="week_start_date",
        columns="geo_region",
        values="observed_session_count"
    )
    .sort_index()
)

transaction_matrix = (
    pre_df.pivot(
        index="week_start_date",
        columns="geo_region",
        values="observed_transaction_count"
    )
    .sort_index()
)

assert revenue_matrix.shape == (8, 20)
assert session_matrix.shape == (8, 20)
assert transaction_matrix.shape == (8, 20)

assert not revenue_matrix.isna().any().any()
assert not session_matrix.isna().any().any()
assert not transaction_matrix.isna().any().any()

print("Revenue matrix:", revenue_matrix.shape)
print("Session matrix:", session_matrix.shape)
print("Transaction matrix:", transaction_matrix.shape)
print("MATCHING MATRICES: PASS")

display(revenue_matrix)

Revenue matrix: (8, 20)
Session matrix: (8, 20)
Transaction matrix: (8, 20)
MATCHING MATRICES: PASS


geo_region,Arizona,California,Colorado,Florida,Georgia,Illinois,Maryland,Massachusetts,Michigan,New Jersey,New York,North Carolina,Ohio,Oregon,Pennsylvania,Tennessee,Texas,Utah,Virginia,Washington
week_start_date,,,,,,,,,,,,,,,,,,,,
2020-11-01,137.0,1782.0,130.0,472.0,636.0,214.0,320.0,1154.0,353.0,240.0,387.0,50.0,35.0,219.0,188.0,134.0,702.0,23.0,121.0,211.0
2020-11-08,22.0,3094.0,38.0,480.0,226.0,485.0,68.0,212.0,358.0,517.0,561.0,409.0,182.0,146.0,22.0,92.0,714.0,82.0,577.0,538.0
2020-11-15,294.0,3558.0,209.0,543.0,502.0,377.0,0.0,660.0,139.0,793.0,407.0,517.0,59.0,50.0,600.0,198.0,1310.0,119.0,551.0,48.0
2020-11-22,141.0,2922.0,654.0,678.0,72.0,661.0,110.0,841.0,273.0,1684.0,1289.0,405.0,464.0,305.0,236.0,279.0,2572.0,218.0,659.0,744.0
2020-11-29,765.0,4927.0,139.0,770.0,430.0,109.0,177.0,387.0,606.0,770.0,707.0,432.0,474.0,321.0,202.0,180.0,2211.0,167.0,1469.0,605.0
2020-12-06,440.0,6211.0,791.0,1549.0,429.0,828.0,245.0,592.0,362.0,386.0,951.0,1509.0,211.0,396.0,727.0,998.0,2125.0,80.0,1587.0,872.0
2020-12-13,1552.0,3687.0,249.0,2220.0,374.0,1022.0,306.0,707.0,1224.0,763.0,1089.0,379.0,413.0,351.0,778.0,112.0,1488.0,458.0,2505.0,493.0
2020-12-20,169.0,1093.0,0.0,372.0,197.0,61.0,0.0,189.0,363.0,71.0,77.0,204.0,0.0,264.0,471.0,41.0,558.0,57.0,404.0,88.0
